In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
training_data = datasets.FashionMNIST(
    root= "data",
    train = True,
    download = True,
    transform = ToTensor(),
)

test_data = datasets.FashionMNIST(
    root= "data",
    train = False,
    download = True,
    transform = ToTensor(),

)


100%|██████████| 26.4M/26.4M [00:01<00:00, 14.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 212kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.2MB/s]


In [ ]:
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size )

for X,y in test_dataloader:
  print(f"Shape of X [N, C, H, W]: {X.shape}")
  print(f"Shape of y [N, C, H, W]: {y.shape}")
  break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y [N, C, H, W]: torch.Size([64])


In [ ]:
from torch.nn.modules.activation import ReLU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"using {device} device")
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512,512),
        nn.ReLU(),
        nn.Linear(512,10)
    )
  def forward(self,x):
      x = self.flatten(x)
      logits = self.linear_relu_stack(x)
      return logits
model = NeuralNetwork().to(device)
print(model)

using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch, (X,y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    pred = model(X)
    loss = loss_fn(pred,y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if(batch%100 ==0):
      loss, current = loss.item(), (batch+1) * len(X)
      print(f"loss: {loss:>7f} [{current:> 5d}/ {size:>5d}]")


In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 1.158609 [   64/ 60000]
loss: 1.155672 [ 6464/ 60000]
loss: 0.979895 [ 12864/ 60000]
loss: 1.122183 [ 19264/ 60000]
loss: 0.997609 [ 25664/ 60000]
loss: 1.023524 [ 32064/ 60000]
loss: 1.059392 [ 38464/ 60000]
loss: 1.006318 [ 44864/ 60000]
loss: 1.042936 [ 51264/ 60000]
loss: 0.972748 [ 57664/ 60000]
Test Error: 
 Accuracy: 65.6%, Avg loss: 0.984955 

Epoch 2
-------------------------------
loss: 1.039319 [   64/ 60000]
loss: 1.056405 [ 6464/ 60000]
loss: 0.865175 [ 12864/ 60000]
loss: 1.027327 [ 19264/ 60000]
loss: 0.908414 [ 25664/ 60000]
loss: 0.929266 [ 32064/ 60000]
loss: 0.980870 [ 38464/ 60000]
loss: 0.931300 [ 44864/ 60000]
loss: 0.963951 [ 51264/ 60000]
loss: 0.904378 [ 57664/ 60000]
Test Error: 
 Accuracy: 66.6%, Avg loss: 0.911349 

Epoch 3
-------------------------------
loss: 0.950827 [   64/ 60000]
loss: 0.987473 [ 6464/ 60000]
loss: 0.783028 [ 12864/ 60000]
loss: 0.960216 [ 19264/ 60000]
loss: 0.849025 [ 25664/ 60000]
loss: 0

In [ ]:
torch.save(model.state_dict(), "dress_classifier_model.pth")
print("Saved the model's state to the dress_classifer_model.pth")

Saved the model's state to the dress_classifer_model.pth


In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("dress_classifier_model.pth"))

<All keys matched successfully>

In [ ]:
 classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
X,y = test_data[0][0], test_data[0][1]
with torch.no_grad():
  x = X.to(device)
  pred = model(x)
  predicted, actual = classes[pred[0].argmax(0)], classes[y]
  print(f'Predicted: "{predicted}", Actual: "{actual}"')


Predicted: "Ankle boot", Actual: "Ankle boot"
